# Merge Half1 + Half2 → Full Match

Trims each half's tracking CSV down to its reviewed/cleaned window, renumbers frames onto a
single continuous match timeline, concatenates the two CSVs, then does the same for the two
videos (frame-accurate trim + concat) so the merged video and merged CSV share one `frame` axis.

**Before running:** `Runtime → Change runtime type → T4 GPU` (only needed for the video cells).

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
H1_CSV   = "/content/per_frame_tracks_half1_unified.csv"
H2_CSV   = "/content/per_frame_tracks_half2_unified.csv"
OUT_CSV  = "/content/per_frame_tracks_full_match.csv"

H1_VIDEO = "/content/half1.mp4"     # raw/annotated half1 source video
H2_VIDEO = "/content/half2.mp4"     # raw/annotated half2 source video
OUT_VIDEO = "/content/full_match.mp4"

# Reviewed/cleaned frame windows (inclusive) — everything outside these is discarded
H1_WINDOW = (1066, 3333)
H2_WINDOW = (4270, 5720)

VIDEO_BITRATE = "8M"   # NVENC bitrate for the trim re-encode
CRF           = 18     # libx264 fallback quality if no NVENC


## 1. Trim + renumber + merge the two CSVs

- Each half is sliced down to its window.
- `frame` is overwritten with a new contiguous index (0-based) spanning the merged timeline;
  the original per-half frame number is preserved in `src_frame` for traceability.
- A `half` column (1 / 2) records which source half each row came from.
- `pass_id` is made globally unique (half2's pass ids are offset past half1's max) so passes
  from the two halves never collide once merged.

In [ ]:
import pandas as pd
import numpy as np

def load_trim(path, window, half):
    df = pd.read_csv(path, low_memory=False)
    lo, hi = window
    df = df[(df["frame"] >= lo) & (df["frame"] <= hi)].copy()
    df["src_frame"] = df["frame"]
    df["half"] = half
    return df.sort_values(["frame", "class_id", "display_track_id"]).reset_index(drop=True)

h1 = load_trim(H1_CSV, H1_WINDOW, half=1)
h2 = load_trim(H2_CSV, H2_WINDOW, half=2)

# contiguous merged frame axis: h1 -> [0 .. n1-1], h2 -> [n1 .. n1+n2-1]
h1_frames = np.sort(h1["frame"].unique())
h2_frames = np.sort(h2["frame"].unique())
h1_map = {f: i for i, f in enumerate(h1_frames)}
n1 = len(h1_frames)
h2_map = {f: n1 + i for i, f in enumerate(h2_frames)}

h1["frame"] = h1["src_frame"].map(h1_map)
h2["frame"] = h2["src_frame"].map(h2_map)

# make pass_id globally unique across halves (offset half2 past half1's max)
if "pass_id" in h1.columns:
    offset = h1["pass_id"].max(skipna=True)
    offset = 0 if pd.isna(offset) else offset
    h2["pass_id"] = h2["pass_id"] + offset

merged = pd.concat([h1, h2], ignore_index=True)
merged = merged.sort_values(["frame", "class_id", "display_track_id"]).reset_index(drop=True)

# sanity: no duplicate (frame, display_track_id) among people classes
people = merged[merged["class_id"].isin([1, 2, 3])]
dup = people.groupby(["frame", "display_track_id"]).size()
assert (dup > 1).sum() == 0, f"{(dup > 1).sum()} duplicate (frame, display_track_id) rows after merge"

merged.to_csv(OUT_CSV, index=False)
print(f"H1: {len(h1_frames)} frames (src {H1_WINDOW[0]}-{H1_WINDOW[1]}) -> merged frame 0-{n1-1}")
print(f"H2: {len(h2_frames)} frames (src {H2_WINDOW[0]}-{H2_WINDOW[1]}) -> merged frame {n1}-{merged['frame'].max()}")
print(f"Merged rows: {len(merged)}  -> saved {OUT_CSV}")


## 2. Trim + concat the two videos

Each video is cut to its window frame-accurately (re-encoded, since container-level cuts only
land on keyframes), then concatenated. Uses NVENC if a GPU is available, otherwise falls back
to libx264.

In [ ]:
# ── GPU / NVENC check ───────────────────────────────────────────────────────
import subprocess

nvenc_ok = False
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                          capture_output=True, text=True)
    if gpu.returncode == 0:
        enc = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True)
        nvenc_ok = "h264_nvenc" in enc.stdout
except FileNotFoundError:
    pass
print("Using encoder:", "h264_nvenc (GPU)" if nvenc_ok else "libx264 (CPU)")


In [ ]:
# ── Frame-accurate trim of each half ───────────────────────────────────────
# If the supplied video is already a pre-trimmed clip (its frame count already matches
# the window length) it is just re-encoded as-is; otherwise it's cut from the full match
# starting at the window's absolute frame offset.
import cv2, os

def fps_of(path):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    cap.release()
    return fps

def frame_count_of(path):
    cap = cv2.VideoCapture(path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return n

def trim_video(src, window, dst):
    lo, hi = window
    expected_len = hi - lo + 1
    actual_len = frame_count_of(src)
    fps = fps_of(src)
    enc = (["-c:v", "h264_nvenc", "-b:v", VIDEO_BITRATE]
           if nvenc_ok else ["-c:v", "libx264", "-crf", str(CRF), "-preset", "medium"])
    if abs(actual_len - expected_len) <= 2:
        print(f"{src}: already trimmed ({actual_len} frames ~= window length {expected_len}), re-encoding only.")
        cmd = ["ffmpeg", "-y", "-i", src, *enc, "-pix_fmt", "yuv420p", "-an", dst]
    else:
        start_t = lo / fps
        cmd = [
            "ffmpeg", "-y", "-ss", f"{start_t:.6f}", "-i", src,
            "-frames:v", str(expected_len),
            *enc, "-pix_fmt", "yuv420p", "-an",
            dst,
        ]
    subprocess.run(cmd, check=True)
    return dst

H1_TRIMMED = "/content/_h1_trimmed.mp4"
H2_TRIMMED = "/content/_h2_trimmed.mp4"
trim_video(H1_VIDEO, H1_WINDOW, H1_TRIMMED)
trim_video(H2_VIDEO, H2_WINDOW, H2_TRIMMED)
print("Trimmed both halves.")


In [ ]:
# ── Concat the two trimmed clips ───────────────────────────────────────────
CONCAT_LIST = "/content/_concat_list.txt"
with open(CONCAT_LIST, "w") as f:
    f.write(f"file '{H1_TRIMMED}'\n")
    f.write(f"file '{H2_TRIMMED}'\n")

enc = (["-c:v", "h264_nvenc", "-b:v", VIDEO_BITRATE]
       if nvenc_ok else ["-c:v", "libx264", "-crf", str(CRF), "-preset", "medium"])
cmd = [
    "ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", CONCAT_LIST,
    *enc, "-pix_fmt", "yuv420p",
    OUT_VIDEO,
]
subprocess.run(cmd, check=True)
for p in (H1_TRIMMED, H2_TRIMMED, CONCAT_LIST):
    os.remove(p)
print(f"Merged video saved -> {OUT_VIDEO}")


## 3. Verify CSV ↔ video alignment

The merged CSV's frame count should match the merged video's frame count (same fps, same
total frame span).

In [ ]:
import pandas as pd, cv2

merged = pd.read_csv(OUT_CSV, low_memory=False)
csv_frames = merged["frame"].nunique()
csv_span = merged["frame"].max() - merged["frame"].min() + 1

cap = cv2.VideoCapture(OUT_VIDEO)
vid_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
vid_fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()

print(f"CSV:   {csv_frames} unique frames (span {csv_span})")
print(f"Video: {vid_frames} frames @ {vid_fps:.2f} fps")
if csv_span != vid_frames:
    print(f"WARNING: mismatch of {abs(csv_span - vid_frames)} frames — check trim boundaries / fps rounding.")
else:
    print("OK: CSV and video frame counts match.")


In [ ]:
# ── Download outputs ───────────────────────────────────────────────────────
from google.colab import files as colab_files
colab_files.download(OUT_CSV)
colab_files.download(OUT_VIDEO)
